# 01 - Cohorts, features, descriptive tables and the confounding design

**What this notebook does**
1. Builds the per-subject tables of both cohorts. The Greek diagnostic labels are read here for the first time; the
   time is logged in outputs/run_log.json and must be later than the plan freeze.
2. Maps the BrainLat BioSemi-128 electrodes to the 19 positions of the 10-20 system available in both cohorts (S1).
3. Extracts the 11 epoch-level features with the harmonised pipeline (cached in outputs/features/). Labels are joined
   only after extraction.
4. Writes Table 1, the domain-shift table (S2), the participant flow (Fig. 0 / F0), cognitive severity by cohort
   (T1b, T1c) and the comparison of included and excluded BrainLat participants (S7).
5. Fixes the confounding design (1:1 education-matched pairs, comparable pairs, overlap weights) **before any
   prediction** (S3).

**Outputs:** outputs/derived/ (per-subject and per-epoch tables used by the next notebooks), outputs/tables/,
outputs/figures/fig0_*.

**Time:** 5-15 min the first time (feature extraction); seconds afterwards (cache).

In [ ]:
# Setup: make the eegad package importable (also when it was not installed with `pip install -e .`)
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=RuntimeWarning)   # MNE notices about head size and 'boundary' events
warnings.filterwarnings("ignore", category=FutureWarning)

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "analysis_plan.json").exists()), None)
if _root is None:
    raise FileNotFoundError("Open this notebook from inside the repository (eeg-ad-demographics/notebooks).")
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

In [ ]:
import hashlib
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from eegad.config import load_context
from eegad.data import (brainlat_files, brainlat_metadata, brainlat_subjects, find_brainlat_root, greek_participants,
                        greek_participants_raw, greek_subjects, load_table, save_table)
from eegad.design import balance, ess, match_1to1, propensity_score, eligible_education, eligible_sex_age
from eegad.features import brainlat_channel_map, extract_cohort, failure_reason
from eegad.figures import apply_style, flow_figure
from eegad.reporting import fmt_ci, save_figure, write_json
from eegad.stats import hedges_g
from eegad.tables import compare_groups, mmse_stages, participants_keys, participants_table, severity_rows

apply_style()
ctx = load_context("01")
log, plan, out = ctx.log, ctx.plan, ctx.out
DERIVED = out / "derived"

## 1) BrainLat: one row per .set file, with demographics and cognition

In [ ]:
brainlat_root = find_brainlat_root(ctx.brainlat_dir, plan)
bl_files = brainlat_files(brainlat_root, plan, log)
demo_all, df_cog, cog_files = brainlat_metadata(brainlat_root, plan)
bl_all = brainlat_subjects(bl_files, demo_all, df_cog, plan)

print(f"BrainLat: {len(bl_all)} .set files ({int(bl_all.y.sum())} AD, {int((1 - bl_all.y).sum())} HC)")
display(pd.crosstab(bl_all["y"].map({1: "AD", 0: "HC"}), bl_all["site"], margins=True))
print(f"With age, sex and education: {int(bl_all[['Age', 'sex', 'years_education']].notna().all(axis=1).sum())}"
      f" | cognition files: {[f.name for f in cog_files]}")
print("Sex code (raw counts by group; 0 = female in the BrainLat dictionary):")
display(pd.crosstab(bl_all["y"].map({1: "AD", 0: "HC"}), bl_all["sex"]))

## 2) Greek cohort (ds004504): AD and cognitively normal controls

Frontotemporal dementia (group F) is excluded by design. derivatives/ files are used (0.5-45 Hz, ASR, ICA), as
distributed by the dataset authors. Education was not recorded (E-value in notebook 03).

In [ ]:
run_log = ctx.mark_greek_labels_read()
gr_raw = greek_participants_raw(ctx.ds004504_dir)
print("Groups in participants.tsv:", gr_raw["group"].value_counts().to_dict())
gr_part, has_education = greek_participants(gr_raw, ctx.ds004504_dir, plan)
gr_all = greek_subjects(gr_part)
print(f"Plan frozen on {run_log['plan_first_frozen_at']} | Greek labels first read on {run_log['greek_labels_first_read']}")
print(f"\nds004504: {len(gr_all)} participants ({int(gr_all.y.sum())} AD, {int((1 - gr_all.y).sum())} controls)")
print("File origin:", gr_all["origin"].value_counts().to_dict())
print(f"Education in the metadata: {'yes' if has_education else 'no (as expected: E-value in notebook 03)'}")
no_file = gr_all[gr_all.file.isna()]
if len(no_file):
    log.alert(f"{len(no_file)} ds004504 participant(s) without a .set file: {no_file.subject_id.tolist()}")

## 3) The 19 BrainLat channels closest to the 10-20 positions (Suppl. Table S1)

Each 10-20 position gets the BioSemi electrode with the smallest angle on the unit sphere (one-to-one assignment).
Positions come from the file itself or, if better, from the MNE biosemi128 template by channel name.

In [ ]:
CHANNELS = ctx.channels
MAP_BL, map_source, montage_name, file0, candidates = brainlat_channel_map(bl_all["file"], CHANNELS)
for k, m in candidates.items():
    print(f"{k}: mean angle {m.angle_deg.mean():.1f}°, maximum {m.angle_deg.max():.1f}°")
display(MAP_BL.round(1))
print(f"Used: {map_source} | 10-20 reference: {montage_name} | file: {Path(file0).name}")
if MAP_BL.angle_deg.max() > 20:
    log.alert("A 10-20 channel is more than 20° from its electrode: check the orientation of the positions.")
MAP_BL.to_csv(out / "tables" / "S1_brainlat_channel_map_1020.csv", index=False)
log.record("Methods", "BioSemi -> 10-20 correspondence",
           f"{map_source}; mean angle {MAP_BL.angle_deg.mean():.1f}°, maximum {MAP_BL.angle_deg.max():.1f}°",
           "Methods; Suppl. Table S1")
log.key("channels.angle_mean", float(MAP_BL.angle_deg.mean()))
log.key("channels.angle_max", float(MAP_BL.angle_deg.max()))

## 4) Harmonised feature extraction (19 channels, 11 features), without labels

Same steps in both cohorts: 19 channels, average reference, 0.5-45 Hz FIR, 4-s epochs, RMS normalisation, Welch at
1 Hz resolution. The Greek recordings are first low-passed at 40 Hz (Butterworth, order 8, zero phase) to match the
band of the distributed BrainLat files. Labels are attached only after extraction.

In [ ]:
map_bl = dict(zip(MAP_BL["channel_brainlat"], MAP_BL["channel_1020"]))
map_gr = {c: c for c in CHANNELS}
base_params = {"channels": CHANNELS, "prep": plan["preprocessing"], "features": ctx.features,
               "max_epochs": ctx.rt["max_epochs"], "version": 1}
items_bl = [(r.subject_id, r.file, map_bl) for r in bl_all.itertuples()]
items_gr = [(r.subject_id, r.file, map_gr) for r in gr_all.itertuples() if isinstance(r.file, str)]
feats_bl, info_bl, fail_bl = extract_cohort(
    items_bl, f"features_19ch_brainlat_{ctx.tag}",
    {**base_params, "mapping": map_bl, "files": [[s, Path(f).name] for s, f, _ in items_bl]}, ctx)
feats_gr, info_gr, fail_gr = extract_cohort(
    items_gr, f"features_19ch_greek_{ctx.tag}",
    {**base_params, "mapping": map_gr, "files": [[s, Path(f).name] for s, f, _ in items_gr],
     "derivatives": plan["cohorts"]["greek"]["use_derivatives"], "band_harmonization": True},
    ctx, harmonize_band=True)
for name, fail in (("BrainLat", fail_bl), ("ds004504", fail_gr)):
    if len(fail):
        print(f"\n{name}: {len(fail)} file(s) without features:")
        display(fail.assign(reason=fail["error"].map(failure_reason))[["subject_id", "reason", "error"]])

# Labels are joined only now
df_bl = feats_bl.merge(bl_all[["subject_id", "y"]], on="subject_id", how="inner").reset_index(drop=True)
df_gr = feats_gr.merge(gr_all[["subject_id", "y"]], on="subject_id", how="inner").reset_index(drop=True)
n_before = {"BrainLat": len(bl_all), "Greek": len(gr_all)}
subj_bl = bl_all[bl_all.subject_id.isin(df_bl.subject_id)].merge(info_bl, on="subject_id", how="left").reset_index(drop=True)
subj_gr = gr_all[gr_all.subject_id.isin(df_gr.subject_id)].merge(info_gr, on="subject_id", how="left").reset_index(drop=True)
for name, s, d, key in (("BrainLat", subj_bl, df_bl, "brainlat"), ("Greek", subj_gr, df_gr, "greek")):
    print(f"{name}: {len(s)} of {n_before[name]} with EEG ({int(s.y.sum())} AD, {int((1 - s.y).sum())} HC) | "
          f"{len(d)} epochs | sampling rate(s) {sorted(set(d.sfreq))} Hz")
    nb_ = s["boundary_marks"].fillna(0)
    if (nb_ > 0).any():
        print(f"  {int((nb_ > 0).sum())} participant(s) with 'boundary' marks (segments cut in the original "
              f"preprocessing; median {nb_[nb_ > 0].median():.0f} per file). Epochs were not dropped.")
        log.record("Methods", f"{name}: files with 'boundary' marks",
                   f"{int((nb_ > 0).sum())} of {len(s)}; median {nb_[nb_ > 0].median():.0f} per file",
                   "Methods (if relevant)")
    log.record("Methods", f"{name}: participants with analysable EEG",
               f"{len(s)} of {n_before[name]} ({int(s.y.sum())} AD, {int((1 - s.y).sum())} HC); {len(d)} epochs",
               "Methods; Table 1")
    log.keys_from(f"cohort.{key}", {"n_analysable": len(s), "n_with_eeg_file": n_before[name],
                                    "n_ad": int(s.y.sum()), "n_hc": int((1 - s.y).sum()), "n_epochs": len(d)})
excluded_bl = sorted(set(bl_files.subject_id) - set(subj_bl.subject_id))
if excluded_bl:
    log.record("Methods", "BrainLat: without analysable EEG", f"{len(excluded_bl)} participant(s) (ids in "
               "outputs/derived/brainlat_files.csv)", "Methods")

# Tables used by the next notebooks (read back with exact float round-trip)
for name, df in (("brainlat_files", bl_all), ("greek_participants", gr_all), ("brainlat_subjects", subj_bl),
                 ("greek_subjects", subj_gr), ("brainlat_epochs", df_bl), ("greek_epochs", df_gr),
                 ("brainlat_demographic_ids", demo_all.assign(y=demo_all["folder_group"].map(plan["labels"]))
                  .drop_duplicates("subject_id")[["subject_id", "y"]]),
                 ("greek_participants_all_groups", gr_raw[["participant_id", "group"]]),
                 ("brainlat_extraction_failures", fail_bl), ("greek_extraction_failures", fail_gr)):
    save_table(df, DERIVED / f"{name}.csv")
subj_bl, subj_gr = load_table(DERIVED / "brainlat_subjects.csv"), load_table(DERIVED / "greek_subjects.csv")
print("\nSaved to", DERIVED)

## 5) Table 1 and domain shift between the cohorts (Suppl. Table S2)

In [ ]:
FEATURES = ctx.features
TAB1 = pd.concat([participants_table(subj_bl, "BrainLat"), participants_table(subj_gr, "Greek (ds004504)")],
                 ignore_index=True)
display(TAB1)
TAB1.to_csv(out / "tables" / "T1_participants.csv", index=False, encoding="utf-8-sig")
for k, v in {**participants_keys(subj_bl, "brainlat"), **participants_keys(subj_gr, "greek")}.items():
    log.key(k, v)

med = (pd.concat([df_bl.assign(cohort="BrainLat"), df_gr.assign(cohort="Greek")])
       .assign(group=lambda d: np.where(d.y == 1, "AD", "HC"))
       .groupby(["cohort", "group"])[FEATURES].median().T)
med.columns = [f"{c} {g}" for c, g in med.columns]
print("Median per epoch (cohort x group):")
display(med.round(3))
smd = {}
for f in FEATURES:
    a = df_bl[df_bl.y == 0].groupby("subject_id")[f].mean()
    b = df_gr[df_gr.y == 0].groupby("subject_id")[f].mean()
    smd[f] = (b.mean() - a.mean()) / np.sqrt((a.var() + b.var()) / 2)
DOMAIN = med.assign(**{"SMD between cohorts (HC, Greek − BrainLat)": pd.Series(smd)})
DOMAIN.to_csv(out / "tables" / "S2_features_by_cohort.csv", encoding="utf-8-sig")
large = [f for f, v in smd.items() if abs(v) > 0.8]
print(f"Features with |SMD| > 0.8 between the controls of the two cohorts: {large or 'none'}")
log.record("Methods", "Domain shift (controls, |SMD| > 0.8 between cohorts)", ", ".join(large) or "none",
           "Results (RQ3); Suppl. Table S2")
for f, v in smd.items():
    log.key(f"domain_shift.smd.{f}", float(v))
log.key("domain_shift.n_features_smd_gt_0.8", len(large))

## 6) Participant flow (Fig. 0, table F0; TRIPOD+AI items 20a and 21)

In [ ]:
COV_BL, COV_GR = ctx.cov_brainlat, ctx.cov_greek
COV_NAMES = {"Age": "age", "sex_fem": "sex", "years_education": "education"}
FLOW = []


def counts(d):
    return {"n": int(len(d)), "AD": int((d["y"] == 1).sum()), "HC": int((d["y"] == 0).sum())}


def reasons(ids, fail):
    if not len(ids):
        return {}
    m = dict(zip(fail["subject_id"].astype(str), fail["error"])) if len(fail) else {}
    return pd.Series([failure_reason(m[i]) if i in m else "no valid 4-s epoch" for i in ids]).value_counts().to_dict()


def step(cohort, key, label, d, excluded=None, reason=None, ftd=np.nan):
    c = counts(d)
    FLOW.append({"cohort": cohort, "key": key, "step": label, **c, "FTD": ftd,
                 "excluded_at_step": "" if excluded is None else excluded,
                 "reason": "" if not reason else "; ".join(f"{k} (n = {v})" for k, v in reason.items())})


demo_ids = load_table(DERIVED / "brainlat_demographic_ids.csv")
no_set = demo_ids[~demo_ids.subject_id.isin(bl_files.subject_id)]
reasons_bl = reasons(excluded_bl, fail_bl)
cc_bl = subj_bl.dropna(subset=COV_BL)
cc_gr = subj_gr.dropna(subset=COV_GR)
covs_txt = lambda covs: ", ".join(COV_NAMES.get(c, c) for c in covs)
step("BrainLat", "demographic_files", "Listed in the distributed demographic files (AD and HC folders)", demo_ids,
     len(no_set), {"no resting-state EEG file (.set)": len(no_set)} if len(no_set) else None)
step("BrainLat", "set_files", "Resting-state EEG file (.set) available", bl_files, len(excluded_bl), reasons_bl)
step("BrainLat", "analysable", "Analysable EEG: within-cohort LOSO (RQ1a, RQ2a) and transport target (RQ3b)",
     subj_bl, len(subj_bl) - len(cc_bl), {"missing covariate": len(subj_bl) - len(cc_bl)} if len(subj_bl) > len(cc_bl) else None)
step("BrainLat", "complete_covariates", f"Complete covariates ({covs_txt(COV_BL)}): incremental-value model", cc_bl)
step("BrainLat", "exploratory", "AD with MoCA (exploratory)", subj_bl[(subj_bl.y == 1) & subj_bl.moca_total.notna()])

grp = gr_raw["group"]
n_ftd = int((~grp.isin(list(plan["cohorts"]["greek"]["groups"]))).sum())
FLOW.append({"cohort": "Greek", "key": "participants_tsv", "step": "Listed in participants.tsv (ds004504 v1.0.9)",
             "n": int(len(gr_raw)), "AD": int((grp == "A").sum()), "HC": int((grp == "C").sum()), "FTD": n_ftd,
             "excluded_at_step": n_ftd, "reason": f"frontotemporal dementia, excluded by design (n = {n_ftd})"})
with_set_gr = gr_all[gr_all.file.notna()]
step("Greek", "ad_and_controls", "AD and cognitively normal controls", gr_all, len(gr_all) - len(with_set_gr),
     {"no .set file": len(gr_all) - len(with_set_gr)} if len(gr_all) > len(with_set_gr) else None)
excluded_gr = sorted(set(with_set_gr.subject_id) - set(subj_gr.subject_id))
step("Greek", "set_files", "Resting-state EEG file (.set) available", with_set_gr, len(excluded_gr),
     reasons(excluded_gr, fail_gr))
step("Greek", "analysable", "Analysable EEG: within-cohort LOSO (RQ1b, RQ2b) and transport target (RQ3a)", subj_gr,
     len(subj_gr) - len(cc_gr), {"missing covariate": len(subj_gr) - len(cc_gr)} if len(subj_gr) > len(cc_gr) else None)
step("Greek", "complete_covariates", f"Complete covariates ({covs_txt(COV_GR)}): incremental-value model", cc_gr)
step("Greek", "exploratory", "AD with MMSE (exploratory)", subj_gr[(subj_gr.y == 1) & subj_gr.mmse.notna()])

TAB_FLOW = pd.DataFrame(FLOW)
TAB_FLOW.drop(columns="key").to_csv(out / "tables" / "F0_participant_flow.csv", index=False, encoding="utf-8-sig")
display(TAB_FLOW.drop(columns="key"))
for r in TAB_FLOW.itertuples():
    log.keys_from(f"flow.{r.cohort.lower()}.{r.key}", {"n": r.n, "ad": r.AD, "hc": r.HC})
fig = flow_figure(TAB_FLOW)
save_figure(fig, "fig0_participant_flow", out, log)
plt.show()
log.record("Submission", "Participant flow (fig0; F0)",
           f"BrainLat {len(bl_files)} with .set -> {len(subj_bl)} analysable ({len(excluded_bl)} excluded: "
           + "; ".join(f"{k} {v}" for k, v in reasons_bl.items()) + f"); Greek {len(gr_raw)} -> {len(gr_all)} AD+HC "
           f"({n_ftd} FTD excluded by design) -> {len(subj_gr)} analysable", "Results (Fig. 1)")

## 7) Cognitive severity by cohort (T1b, T1c) and who was excluded in BrainLat (S7)

- MoCA and IFS (BrainLat) and MMSE (Greek): n, mean ± SD, median [IQR] and range in AD and controls, and Hedges' g
  (AD − HC, 95% CI): how far the AD group is from its own controls, in SDs. This is the severity comparison that works
  across cohorts that used different scales; MoCA and MMSE are not interchangeable.
- Greek AD by MMSE stage (Perneczky et al., 2006: 21-25 mild, 11-20 moderate, 0-10 severe).
- Excluded vs included BrainLat participants of the same diagnostic group (descriptive).

In [ ]:
SEV = []
for cohort, key, s, col, label in (("BrainLat", "brainlat", subj_bl, "moca_total", "MoCA (0–30)"),
                                   ("BrainLat", "brainlat", subj_bl, "ifs_total_score", "IFS (0–30)"),
                                   ("Greek (ds004504)", "greek", subj_gr, "mmse", "MMSE (0–30)")):
    if col not in s or s[col].notna().sum() == 0:
        continue
    rows, g = severity_rows(cohort, s, col, label)
    SEV += rows
    scale = label.split(" ")[0].lower()
    log.keys_from(f"severity.{key}.{scale}.hedges_g", dict(zip(("value", "lo", "hi"), g)))
    log.record("Submission", f"{cohort}: severity ({label.split(' ')[0]}), Hedges g AD − HC",
               f"{fmt_ci(*g, 2)}; AD {rows[0]['mean ± SD']} (n = {s.loc[s.y == 1, col].notna().sum()}), "
               f"HC {rows[1]['mean ± SD']} (n = {s.loc[s.y == 0, col].notna().sum()})", "Table 1b; Limitations")
TAB_SEV = pd.DataFrame(SEV)
TAB_SEV.to_csv(out / "tables" / "T1b_severity_by_cohort.csv", index=False, encoding="utf-8-sig")
display(TAB_SEV)

TAB_STAGE, n_mm = mmse_stages(subj_gr.loc[subj_gr.y == 1, "mmse"])
TAB_STAGE.to_csv(out / "tables" / "T1c_greek_mmse_stages.csv", index=False, encoding="utf-8-sig")
print(f"Greek AD with MMSE (n = {n_mm}), stages of Perneczky et al. (2006):")
display(TAB_STAGE)
for _, r in TAB_STAGE.iterrows():
    log.key(f"severity.greek.mmse_stage.{r['stage (MMSE)'].split(' ')[0]}", int(r["n AD"]))
if n_mm:
    log.record("Submission", "Greek: AD by MMSE stage (Perneczky 2006)",
               "; ".join(f"{r['stage (MMSE)']}: {r['n AD']} ({r['%']}%)" for _, r in TAB_STAGE.iterrows()),
               "Table 1b; Discussion (spectrum)")

# Included vs excluded, within the same diagnostic group (an excluded control is compared with included controls)
bl_all["included"] = bl_all.subject_id.isin(subj_bl.subject_id)
parts = []
for y, grp_name in ((0, "HC"), (1, "AD")):
    I, E = bl_all[(bl_all.y == y) & bl_all.included], bl_all[(bl_all.y == y) & ~bl_all.included]
    if len(E):
        parts.append(compare_groups(I, E).assign(group=grp_name))
excl = bl_all[~bl_all.included]
TAB_EXCL = (pd.concat(parts, ignore_index=True)[["group", "variable", "included", "excluded", "p", "test"]]
            if parts else pd.DataFrame({"group": ["—"], "variable": ["n excluded"], "included": [len(bl_all)],
                                        "excluded": [0], "p": [""], "test": [""]}))
TAB_EXCL.to_csv(out / "tables" / "S7_brainlat_included_vs_excluded.csv", index=False, encoding="utf-8-sig")
print(f"BrainLat: excluded participants by group and site: "
      f"{excl.groupby(['y', 'site']).size().rename(index={0: 'HC', 1: 'AD'}, level=0).to_dict() if len(excl) else 'none'}")
display(TAB_EXCL)
if len(excl):
    log.record("Submission", "BrainLat: who was excluded (no analysable EEG)",
               "; ".join(f"{'AD' if y == 1 else 'HC'} {s}: {n}" for (y, s), n in excl.groupby(["y", "site"]).size().items()),
               "Methods; Limitations")

## 8) Confounding design, fixed before any prediction (Suppl. Table S3)

- BrainLat: optimal 1:1 matching on education (caliper 2 years, ties broken by age); comparable pairs (education
  within 2 years); overlap weights for age, sex and education.
- Greek: comparable pairs (same sex, age within 2 years); overlap weights for age and sex.

The pairs and the design summary are saved now; notebook 04 checks that they did not change.

In [ ]:
R2 = plan["rq2"]
CAL = R2["brainlat_matching"]["caliper_years"]
VAR_BAL_BL = {"years_education": "Education (years)", "Age": "Age (years)", "sex_fem": "Female (proportion)",
              "site_cl": "Chile (proportion)"}
VAR_BAL_GR = {"Age": "Age (years)", "sex_fem": "Female (proportion)"}

pairs_bl = match_1to1(subj_bl, CAL)
N_PAIRS = len(pairs_bl)
matched = subj_bl[subj_bl.subject_id.isin(set(pairs_bl.ad_id) | set(pairs_bl.hc_id))]
print(f"BrainLat — 1:1 pairs with |education difference| <= {CAL}: {N_PAIRS} ({2 * N_PAIRS} of {len(subj_bl)} subjects)")
BAL = {"BrainLat matching": balance(subj_bl, VAR_BAL_BL).merge(balance(matched, VAR_BAL_BL), on="variable",
                                                               suffixes=("_all", "_matched"))}
f_pairs = DERIVED / "design_pairs_brainlat.csv"
pairs_bl.to_csv(f_pairs, index=False)

elig_bl = eligible_education(R2["brainlat_comparable_pairs"]["education_caliper_years"])
elig_gr = eligible_sex_age(R2["greek_comparable_pairs"]["age_caliper_years"])
tbl_c = subj_bl.dropna(subset=["years_education"])
n_comp_bl = int(np.asarray(elig_bl(tbl_c[tbl_c.y == 1], tbl_c[tbl_c.y == 0])).sum())
tgr_c = subj_gr.dropna(subset=["Age", "sex_fem"])
n_comp_gr = int(np.asarray(elig_gr(tgr_c[tgr_c.y == 1], tgr_c[tgr_c.y == 0])).sum())
print(f"Comparable pairs — BrainLat (education ±2 y): {n_comp_bl} | Greek (same sex, age ±"
      f"{R2['greek_comparable_pairs']['age_caliper_years']} y): {n_comp_gr}")

DESIGN = {"generated_at": pd.Timestamp.now().isoformat(timespec="seconds"),
          "note": "generated in notebook 01, before any prediction (notebook 02)",
          "n_pairs_1to1": N_PAIRS, "pairs_sha1": hashlib.sha1(f_pairs.read_bytes()).hexdigest()[:12],
          "n_comparable_pairs_brainlat": n_comp_bl, "n_comparable_pairs_greek": n_comp_gr}
for name, s, covs, vb in (("BrainLat", subj_bl, R2["brainlat_overlap"], VAR_BAL_BL),
                          ("Greek", subj_gr, R2["greek_overlap"], VAR_BAL_GR)):
    t = s.dropna(subset=covs).reset_index(drop=True)
    ps, method = propensity_score(t, covs)
    w = np.where(t["y"] == 1, 1 - ps, ps)
    yy = t["y"].to_numpy(int)
    BAL[f"{name} overlap"] = balance(t, vb).merge(balance(t, vb, w), on="variable", suffixes=("_raw", "_weighted"))
    DESIGN[f"overlap_{name}"] = {"covariates": covs, "ps_method": method, "ess_AD": ess(w[yy == 1]),
                                 "ess_HC": ess(w[yy == 0]), "n": len(t)}
    print(f"{name} — overlap weights ({' + '.join(covs)}; {method}): effective size AD {ess(w[yy == 1]):.1f} of "
          f"{int(yy.sum())}, HC {ess(w[yy == 0]):.1f} of {int((1 - yy).sum())}")
    log.keys_from(f"design.overlap.{name.lower()}", {"ess_ad": ess(w[yy == 1]), "ess_hc": ess(w[yy == 0]), "n": len(t)})
for k, b in BAL.items():
    print(f"\nBalance — {k}:")
    display(b.round(3))
    b.to_csv(out / "tables" / f"S3_balance_{k.replace(' ', '_')}.csv", index=False, encoding="utf-8-sig")
    prefix = {"BrainLat matching": "balance.brainlat_matching", "BrainLat overlap": "balance.brainlat_overlap",
              "Greek overlap": "balance.greek_overlap"}[k]
    for _, r in b.iterrows():
        var = r["variable"].split(" (")[0].lower().replace(" ", "_")
        for col in [c for c in b.columns if c.startswith("SMD")]:
            log.key(f"{prefix}.{var}.{col.replace('SMD_', 'smd_')}", float(r[col]))
write_json(out / "tables" / "confounding_design.json", DESIGN)
log.keys_from("design", {"matched_pairs": N_PAIRS})
log.key("design.comparable_pairs.brainlat", n_comp_bl)
log.key("design.comparable_pairs.greek", n_comp_gr)
if N_PAIRS < R2["brainlat_matching"]["min_pairs"]:
    log.alert(f"Only {N_PAIRS} 1:1 pairs (< {R2['brainlat_matching']['min_pairs']}): the matched AUC is descriptive only.")
print("\nNext: notebook 02.")